In [1]:
import os

import cv2
import torch
import torch_tensorrt
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import mss
import numpy as np
import math
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import mediapipe as mp

from detectors import DETECTOR

import yaml
import time

from collections import deque

import joblib
import numpy as np

Neither TRTLLM_PLUGIN_PATH is set nor is it directed to download the shared library. Please set either of the two to use TRT-LLM libraries in torchTRT


[02/24/2026-17:40:01] [TRT] [W] Functionality provided through tensorrt.plugin module is experimental.


OSError: Could not load this library: /home/antoine/miniconda3/envs/df40/lib/python3.12/site-packages/torch_tensorrt/lib/libtorchtrt.so

In [3]:
def load_config(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

In [ ]:
config = load_config("../config/detector/clip_base.yaml")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

ckpt = torch.load("../weights/train_on_df40-all-ff/clip.pth", map_location=device)

if 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']

new_weights = {}

for key, value in ckpt.items():
    new_key = key.replace('module.', '')
    if 'base_model.' in new_key:
        new_key = new_key.replace('base_model.', 'backbone.')
    if 'classifier.' in new_key:
        new_key = new_key.replace('classifier.', 'head.')
    if 'HRNet_layer.' in new_key:
        new_key = new_key.replace('HRNet_layer.', 'backbone.')
    new_weights[new_key] = value

model_class = DETECTOR[config['model_name']]
model_clip = model_class(config).to(device).half()

model_clip.load_state_dict(new_weights, strict=True)
print('===> Load checkpoint done for clip!')

model_clip.eval().to(device)

In [ ]:
inputs = [
    torch_tensorrt.Input(
        min_shape=[1, 3, 224, 224],
        opt_shape=[1, 3, 224, 224],
        max_shape=[1, 3, 224, 224],
        dtype=torch.float16,
    )
]

In [ ]:
trt_clip = torch_tensorrt.compile(
    model_clip,
    inputs=inputs,
    enabled_precisions={torch.float16}, # Forcer l'utilisation des coeurs Tensor FP16
    workspace_size=1 << 30 # Autoriser 1 Go de VRAM pour la compilation
)